# Lecture 9 · Agent systems: control, recovery and evaluation (Fable edition)

Companion notebook, generated from the code on the lecture pages (https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-09-fable/). Run the cells in order; each one corresponds to a numbered step on the pages. Steps marked 💳 need `LLM_API_KEY` (and optionally `LLM_BASE_URL`, `LLM_MODEL`) in the environment and are skipped otherwise.

## Step 1 — Data, a scripted model, and an action policy that code enforces

[1. Decide what the assistant may do on its own](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-09-fable/01-decide-what-the-assistant-may-do-on-its-own/)

In [ ]:
# The assistant's data, a scripted model, and an action policy that code
# enforces.
import json, os, random, re, sys, time, pathlib, sqlite3, urllib.request
from collections import Counter, defaultdict
from typing import Literal
from pydantic import BaseModel, Field
from sklearn.feature_extraction.text import TfidfVectorizer

SITE = "https://aiml-study-guides.onrender.com"
def fetch(name, url):
    path = pathlib.Path(name)
    if not path.exists():
        urllib.request.urlretrieve(SITE + url, path)
    return [json.loads(line) for line in path.open(encoding="utf-8")]

# The frozen course assets: Lecture 8's labelled questions (same split) and the
# notes corpus.
rows = fetch("l8-questions.jsonl",
             "/datasets/generative-agentic-ai/l8-questions.jsonl")
notes = [
    r for r in fetch("lectures-01-06.en.jsonl",
                     "/corpus/generative-agentic-ai/lectures-01-06.en.jsonl")
    if r["heading"]
]
for i, r in enumerate(rows):
    r["family"] = r["section"] if r["lecture"] else f"none-{i}"
random.seed(0)
families = sorted({r["family"] for r in rows}); random.shuffle(families)
fifth = len(families) // 5
test_f, val_f = set(families[:fifth]), set(families[fifth:2 * fifth])
# 145 questions, used throughout
val = [r for r in rows if r["family"] in val_f]
# stands in for "production" traffic (Step 11)
train = [r for r in rows if r["family"] not in test_f | val_f]

vec = TfidfVectorizer(stop_words="english").fit([n["text"] for n in notes])
doc_m = vec.transform([n["text"] for n in notes])
def retrieve(question, k=3):
    scores = (doc_m @ vec.transform([question]).T).toarray().ravel()
    return [notes[i] for i in scores.argsort()[::-1][:k]]

# A scripted chat model: no key, the same reply every time. Asked for a schema,
# it routes to the lecture
# of the first note in the prompt; otherwise it answers with that note's first
# sentence. It counts calls.
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage
from langchain_core.outputs import ChatGeneration, ChatResult
from langchain_core.utils.function_calling import convert_to_openai_tool

class ScriptedModel(BaseChatModel):
    calls: int = 0

    def _generate(self, messages, stop=None, run_manager=None, tool=None,
                  **kwargs):
        self.calls += 1
        text = str(messages[-1].content)
        found = re.search(r"\[Lecture (\d)\]", text)
        if tool:
            reply = AIMessage(
                "", tool_calls=[{
                    "name": tool,
                    "args": {"lecture": int(found.group(1)) if found else 0},
                    "id": f"c{self.calls}"
                }])
        else:  # the first plain sentence of the notes in the prompt
            body = re.sub(r"\[Lecture \d\][^\n]*\n", " ",
                          text.split("\n\nQuestion:")[0])
            sentences = re.split(r"(?<=[.!?])\s+", body.strip())
            reply = AIMessage(
                next((x for x in sentences if 8 <= len(x.split()) <= 40 and
                      x[0].isupper() and not set("|·#") & set(x)),
                     "The notes do not cover this."))
        return ChatResult(generations=[ChatGeneration(message=reply)])
    def bind_tools(self, tools, **kwargs):
        return self.bind(
            tool=convert_to_openai_tool(tools[0])["function"]["name"])
    @property
    def _llm_type(self):
        return "scripted"

# How much the assistant may do alone is decided per tool, from what a wrong
# call would cost.
# harm of a wrong call; can the student undo it; does it speak to other people
# for the student
TOOLS = {
    "search_notes":   dict(harm="none", undo=True,  others=False),
    "route_question": dict(harm="none", undo=True,  others=False),
    "add_todo":       dict(harm="low",  undo=True,  others=False),
    "delete_todo":    dict(harm="low",  undo=False, others=False),
    # an email to the student, cannot be unsent
    "send_reminder":  dict(harm="high", undo=False, others=False),
    # a message in the student's name
    "email_tutor":    dict(harm="high", undo=False, others=True),
}
def decide(t):
    if t["harm"] == "none":
        return "auto"        # act; it shows in the log
    if t["others"]:
        return "suggest"     # draft only; the student sends it
    if t["undo"]:
        return "report"      # act, then tell the student, who can undo it
    return "approve"         # pause; a person approves, edits or rejects first
POLICY = {name: decide(t) for name, t in TOOLS.items()}
USER_ROLE = {
    "auto": "observer", "report": "observer", "approve": "approver",
    "suggest": "operator"
}
print(f"{'tool':<16}{'harm':<6}{'undo':<6}{'others':<8}"
      f"{'policy':<9}student's role")
for name, t in TOOLS.items():
    print(
        f"{name:<16}{t['harm']:<6}{str(t['undo']):<6}"
        f"{str(t['others']):<8}{POLICY[name]:<9}{USER_ROLE[POLICY[name]]}"
    )

# The guard: code between the model's request and the tool. The model may ask
# for anything.
def guard(call, approved=()):
    # an unknown tool is treated as risky
    rule = POLICY.get(call["name"], "approve")
    if rule in ("auto", "report") or call["id"] in approved:
        return "ran" + (" + notice" if rule == "report" else "")
    return "drafted" if rule == "suggest" else "held for approval"

# What a model asked for on six requests (scripted: the same calls every run).
asked = [
    ("What does LoRA change in a model?", [("search_notes", {"query": "LoRA"})
                                           ]),
    ("Add 'read Lecture 7' to my list", [("add_todo",
                                          {"text": "read Lecture 7"})]),
    ("Delete the bread task", [("delete_todo", {"id": 2})]),
    ("Remind me about the quiz at 9 tomorrow", [("send_reminder",
                                                 {"todo": 3, "at": "09:00"})]),
    ("Tell my tutor my homework is late",
     [("email_tutor", {"text": "My homework will be late."})]),
    ("Clean up my list", [("delete_todo", {"id": i}) for i in (1, 2, 3)]),
]
outcome = Counter()
for request, calls in asked:
    results = [
        guard({"name": n, "args": a, "id": f"{request[:8]}-{j}"})
        for j, (n, a) in enumerate(calls)
    ]
    outcome.update(results)
    print(f"{request:<40}→ " + ", ".join(f"{n}: {r}"
                                         for (n, _), r in zip(calls, results)))
print(f"{sum(len(c) for _, c in asked)} calls asked for: " +
      ", ".join(f"{v} {k}" for k, v in outcome.items()))

# The hosted model for the 💳 steps (8–10): provider-neutral settings from the
# environment, never a key in code.
LLM_READY = bool(os.environ.get("LLM_API_KEY"))
MODEL = os.environ.get("LLM_MODEL", "gpt-6-luna")
# $ per 1M tokens, gpt-6-luna, checked 2026-11-01
PRICE_IN, PRICE_OUT = 0.10, 0.50
if LLM_READY:
    from langchain_openai import ChatOpenAI
    hosted = ChatOpenAI(model=MODEL, api_key=os.environ["LLM_API_KEY"],
                        base_url=os.environ.get("LLM_BASE_URL"),
                        use_responses_api=True, reasoning={"effort": "none"},
                        max_retries=4)
print("hosted model:",
      MODEL if LLM_READY else "not configured; the 💳 steps will be skipped")

## Step 2 — Pause for approval, end the process, resume later

[2. Stop for a person, and resume later](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-09-fable/02-stop-for-a-person-and-resume-later/)

In [ ]:
# Pause a run for a person's decision, end the process, resume later from the
# saved state.
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt, Command
from langgraph.checkpoint.sqlite import SqliteSaver

class Job(TypedDict, total=False):
    request: str
    calls: list          # the tool calls the model asked for
    decisions: dict      # call id → "approve" or "reject", given by a person
    results: list

PLANS = dict(asked)      # the scripted model's requests from Step 1
def plan(s):
    return {
        "calls": [{"name": n, "args": a, "id": f"call-{j}"}
                  for j, (n, a) in enumerate(PLANS[s["request"]])]
    }

def review(s):
    pending = [c for c in s["calls"] if POLICY[c["name"]] == "approve"]
    if not pending:
        return {"decisions": {}}
    # the run stops here
    answer = interrupt(
        {"question": "Approve these actions?", "calls": pending})
    return {"decisions": answer}  # resumes here with the answer

def act(s):
    results = []
    for c in s["calls"]:
        rule = POLICY[c["name"]]
        if rule == "approve" and s["decisions"].get(c["id"]) != "approve":
            results.append(f"{c['name']} {c['args']}: not done "
                           f"({s['decisions'].get(c['id'], 'no answer')})")
        else:
            results.append(f"{c['name']} {c['args']}: " +
                           ("drafted" if rule == "suggest" else "done"))
    return {"results": results}

def build_job_graph():
    g = StateGraph(Job)
    g.add_node(plan); g.add_node(review); g.add_node(act)
    g.add_edge(START, "plan"); g.add_edge("plan", "review")
    g.add_edge("review", "act"); g.add_edge("act", END)
    return g

DB = "l9-jobs.sqlite"
pathlib.Path(DB).unlink(missing_ok=True)
# what a fresh process does: connect to the database, compile the graph
def open_app():
    conn = sqlite3.connect(DB, check_same_thread=False)
    return build_job_graph().compile(checkpointer=SqliteSaver(conn)), conn

# Monday, 18:00: the request arrives; the run reaches the pause.
app, conn = open_app()
cfg = {"configurable": {"thread_id": "student-42/remind-quiz"}}
out = app.invoke({"request": "Remind me about the quiz at 9 tomorrow"}, cfg,
                 version="v2")
print("paused, asking:", out.interrupts[0].value)
print("state so far:", sorted(out.value), "| next node:",
      app.get_state(cfg).next)
conn.close(); del app  # the process ends; only the database file remains
print(f"process ended; {DB} holds {pathlib.Path(DB).stat().st_size:,} bytes")

# Monday, 21:40: the student answers in the app; a new process handles it.
app, conn = open_app()
snap = app.get_state(cfg)
print("found the paused run:", snap.next, "| waiting for:",
      [c["name"] for c in snap.interrupts[0].value["calls"]])
out = app.invoke(Command(resume={"call-0": "approve"}), cfg, version="v2")
print("resumed:", out.value["results"], "| pauses left:", len(out.interrupts))

# The same answer sent with a thread id made up on the spot finds nothing to
# resume.
lost = {"configurable": {"thread_id": f"student-42/{random.randrange(10**6)}"}}
print("new thread id, saved state:",
      app.get_state(lost).values, "| next:",
      app.get_state(lost).next)
try:
    out = app.invoke(Command(resume={"call-0": "approve"}), lost, version="v2")
    print("new thread id, result:", out.value)
except Exception as e:
    print("new thread id:", type(e).__name__, str(e)[:120])
print("the original thread is still complete:",
      app.get_state(cfg).values["results"])

# A rejection: the run continues without the action.
cfg2 = {"configurable": {"thread_id": "student-42/clean-up"}}
out = app.invoke({"request": "Clean up my list"}, cfg2, version="v2")
print("asking about", len(out.interrupts[0].value["calls"]), "deletions")
out = app.invoke(Command(resume={"call-0": "approve", "call-1": "reject"}),
                 cfg2, version="v2")
for line in out.value["results"]:
    print("  ", line)
conn.close()

## Step 3 — What runs again after a resume or a retry

[2. Stop for a person, and resume later](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-09-fable/02-stop-for-a-person-and-resume-later/)

In [ ]:
# What runs again after a resume or a retry, and how to make a repeated action
# harmless.
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import RetryPolicy

class MailService:  # stands in for an email provider outside the graph
    def __init__(self):
        self.sent, self.keys = [], set()
    def send(self, text, key=None):
        # the provider has seen this key: do nothing again
        if key is not None and key in self.keys:
            return "duplicate ignored"
        self.keys.add(key); self.sent.append(text)
        return "sent"

class Flow(TypedDict, total=False):
    ok: bool
    status: str

# 1) A side effect placed before interrupt() in the same node runs again when
# the node resumes.
def run_pause(build):
    mail = MailService()
    app = build(mail).compile(checkpointer=InMemorySaver())
    cfg = {"configurable": {"thread_id": "t1"}}
    app.invoke({}, cfg)
    app.invoke(Command(resume=True), cfg)
    return mail.sent

def same_node(mail):
    def review(s):
        # runs on the first pass…
        mail.send("Push: please approve your quiz reminder")
        # …stops here, and on resume the node starts over
        ok = interrupt("approve the reminder?")
        return {"ok": ok}
    g = StateGraph(Flow); g.add_node(review); g.add_edge(START, "review")
    g.add_edge("review", END)
    return g

def own_node(mail):
    def notify(s):
        # a finished node is saved and never repeated
        mail.send("Push: please approve your quiz reminder")
        return {}
    def review(s):
        return {"ok": interrupt("approve the reminder?")}
    g = StateGraph(Flow); g.add_node(notify); g.add_node(review)
    g.add_edge(START, "notify"); g.add_edge("notify", "review")
    g.add_edge("review", END)
    return g

print("push before interrupt, same node:", len(run_pause(same_node)), "sent")
print("push in its own node before it: ", len(run_pause(own_node)), "sent")

# 2) A retry repeats a node whose action may already have happened: the email
# went out,
#    then the connection dropped before the provider's reply arrived.
def run_retry(with_key):
    mail, drops = MailService(), [True]
    def send_reminder(s):
        # thread id + tool call id
        key = "student-42/remind-quiz/call-0" if with_key else None
        status = mail.send("Reminder: quiz at 9:00", key=key)
        if drops:  # the reply is lost, once
            drops.pop()
            raise ConnectionError("no reply from the mail provider")
        return {"status": status}
    g = StateGraph(Flow)
    g.add_node(
        send_reminder,
        retry_policy=RetryPolicy(max_attempts=3, initial_interval=0.01,
                                 retry_on=ConnectionError))
    g.add_edge(START, "send_reminder"); g.add_edge("send_reminder", END)
    out = g.compile().invoke({})
    return len(mail.sent), out["status"]

for with_key in (False, True):
    n, status = run_retry(with_key)
    print(f"retry after a lost reply, idempotency key "
          f"{'yes' if with_key else 'no '}: {n} email(s) sent, "
          f"last call: {status}")

# 3) Two branches run in one step; one fails. Its finished sibling's result is
# kept (a pending write)
#    and the resumed run does not repeat it.
runs, fail_once = Counter(), [True]
class Pair(TypedDict, total=False):
    found: list
    free: str
def search_notes(s):
    runs["search_notes"] += 1
    return {"found": [n["heading"] for n in retrieve("quiz on LoRA", k=2)]}
def check_calendar(s):
    runs["check_calendar"] += 1
    if fail_once:
        fail_once.pop()
        raise TimeoutError("calendar did not answer")
    return {"free": "Tuesday 9:00"}
def merge(s):
    runs["merge"] += 1
    return {}
g = StateGraph(Pair)
for f in (search_notes, check_calendar, merge):
    g.add_node(f)
g.add_edge(START, "search_notes"); g.add_edge(START, "check_calendar")
g.add_edge(["search_notes", "check_calendar"], "merge")
g.add_edge("merge", END)
app = g.compile(checkpointer=InMemorySaver())
cfg = {"configurable": {"thread_id": "t2"}}
try:
    app.invoke({}, cfg)
except TimeoutError as e:
    print("first run failed:", e, "| runs so far:", dict(runs))
snap = app.get_state(cfg)
print(
    "saved:", {
        t.name: ("result kept" if t.result is not None else "to run")
        for t in snap.tasks
    })
out = app.invoke(None, cfg)                  # resume from the last checkpoint
print("resumed:", out["free"], "|", len(out["found"]), "notes | runs:",
      dict(runs))

## Step 4 — History, replay and a fork

[2. Stop for a person, and resume later](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-09-fable/02-stop-for-a-person-and-resume-later/)

In [ ]:
# Every checkpoint of a thread is kept: list them, replay from one, or fork
# with a corrected state.
class Route(BaseModel):
    """The lecture of the course notes that answers the question."""
    lecture: Literal[0, 1, 2, 3, 4, 5, 6] = Field(
        description="1–6: the lecture that answers it; 0: no lecture does")

def as_context(found):
    return "\n\n".join(f"[Lecture {n['lecture']}] "
                       f"{n['heading']}\n{' '.join(n['text'].split()[:120])}"
                       for n in found)
def in_lecture(question, lecture, k=3):
    scores = (doc_m @ vec.transform([question]).T).toarray().ravel()
    return [
        notes[i] for i in scores.argsort()[::-1]
        if notes[i]["lecture"] == lecture
    ][:k]

class Turn(TypedDict, total=False):
    question: str
    notes: str
    route: int
    answer: str

def build_qa_graph(model):
    def get_notes(s):
        return {"notes": as_context(retrieve(s["question"]))}
    def route(s):
        reply = model.with_structured_output(Route).invoke([
            ("system", "Route the question to a lecture, 0 if none."),
            ("human", f"{s['notes']}\n\nQuestion: {s['question']}")
        ])
        return {"route": reply.lecture}
    def answer(s):
        if s["route"] == 0:
            return {"answer": "The course notes do not cover this."}
        context = as_context(in_lecture(s["question"], s["route"]))
        return {
            "answer": model.invoke([
                ("human", f"{context}\n\nQuestion: {s['question']}")
            ]).content
        }
    g = StateGraph(Turn)
    g.add_node(get_notes); g.add_node(route); g.add_node(answer)
    g.add_edge(START, "get_notes"); g.add_edge("get_notes", "route")
    g.add_edge("route", "answer"); g.add_edge("answer", END)
    return g

model = ScriptedModel()
qa = build_qa_graph(model).compile(checkpointer=InMemorySaver())
cfg = {"configurable": {"thread_id": "q-17"}}
# A validation question the scripted router sends to the wrong lecture.
case = next(q for q in val if q["lecture"] and build_qa_graph(
    ScriptedModel()).compile().invoke(q)["route"] != q["lecture"])
out = qa.invoke({"question": case["question"]}, cfg)
print(f"question: {case['question']}\nroute {out['route']} (correct: "
      f"{case['lecture']}), {model.calls} model calls | "
      f"answer: {out['answer'][:80]}")

history = list(qa.get_state_history(cfg))            # newest first
print(f"{len(history)} checkpoints:")
for snap in reversed(history):
    print(f"  step {snap.metadata['step']:>2}  next {str(snap.next):<16} keys "
          f"{sorted(snap.values)}")

# Replay: run again from the checkpoint taken just before "answer". Earlier
# nodes are not repeated.
before_answer = next(s for s in history if s.next == ("answer",))
calls = model.calls
qa.invoke(None, before_answer.config)
print(f"replay from before 'answer': {model.calls - calls} model call(s)")

# Fork: write a corrected route into that checkpoint. This adds a new
# checkpoint; the old ones stay.
fork_cfg = qa.update_state(before_answer.config, {"route": case["lecture"]})
out = qa.invoke(None, fork_cfg)
print(f"fork with route {case['lecture']}: {out['answer'][:90]}")
print(
    f"checkpoints in the thread now: {len(list(qa.get_state_history(cfg)))}; "
    f"the original answer is still at its checkpoint: "
    f"{history[0].values['route']} → {history[0].values['answer'][:40]}…")

## Step 5 — Route with Command, fan out with Send

[3. Shape the workflow](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-09-fable/03-shape-the-workflow/)

In [ ]:
# Command: update the state and choose the next node in one return. Send: one
# branch per item, merged by a reducer.
import operator
from typing import Annotated
from langgraph.types import Send
from langgraph.errors import InvalidUpdateError

class Ticket(TypedDict, total=False):
    request: str
    kind: str
    # every node that ran appends its name
    handled_by: Annotated[list, operator.add]

def build_ticket_graph(leftover_edge):
    def classify(s) -> Command[Literal["answer", "escalate"]]:
        kind = "urgent" if re.search(r"\b(today|deadline|tonight)\b",
                                     s["request"]) else "question"
        return Command(update={"kind": kind, "handled_by": ["classify"]},
                       goto="escalate" if kind == "urgent" else "answer")
    def answer(s):
        return {"handled_by": ["answer"]}
    def escalate(s):
        return {"handled_by": ["escalate"]}
    g = StateGraph(Ticket)
    g.add_node(classify); g.add_node(answer); g.add_node(escalate)
    g.add_edge(START, "classify")
    if leftover_edge:
        # left over from before classify chose the route itself
        g.add_edge("classify", "answer")
    g.add_edge("answer", END); g.add_edge("escalate", END)
    return g.compile()

urgent = {
    "request": "The homework deadline is tonight and the notebook "
    "will not open"
}
print("with a leftover static edge:",
      build_ticket_graph(True).invoke(urgent)["handled_by"])
print("Command alone:              ",
      build_ticket_graph(False).invoke(urgent)["handled_by"])

# Send: a question about two topics becomes two look-ups that run in the same
# step.
class Compare(TypedDict, total=False):
    question: str
    topics: list
    # the reducer: results from parallel branches are concatenated
    found: Annotated[list, operator.add]
    answer: str

def build_compare_graph(state_type):
    def split(s):
        return {
            "topics": re.findall(r"\b(LoRA|RAG|DPO|quantization)\b",
                                 s["question"])
        }
    def fan_out(s):
        return [
            Send("look_up", {"question": s["question"], "topics": [t]})
            for t in s["topics"]
        ]
    def look_up(s):
        top = retrieve(s["topics"][0], k=1)[0]
        return {
            "found": [
                f"{s['topics'][0]} → "
                f"Lecture {top['lecture']}, {top['heading']}"
            ]
        }
    def combine(s):
        return {"answer": " | ".join(s["found"])}
    g = StateGraph(state_type)
    g.add_node(split); g.add_node(look_up); g.add_node(combine)
    g.add_edge(START, "split")
    g.add_conditional_edges("split", fan_out, ["look_up"])
    g.add_edge("look_up", "combine"); g.add_edge("combine", END)
    return g.compile()

q = {"question": "Compare LoRA with RAG: when should the assistant use each?"}
for event in build_compare_graph(Compare).stream(q, stream_mode="debug"):
    if event["type"] == "task_result":
        print(f"  super-step {event['step']}: {event['payload']['name']:<8} "
              f"{dict(event['payload']['result'])}")

class CompareNoReducer(TypedDict, total=False):
    question: str
    topics: list
    found: list  # no reducer: two writes in one step conflict
    answer: str
try:
    build_compare_graph(CompareNoReducer).invoke(q)
except InvalidUpdateError as e:
    print("without a reducer:", type(e).__name__, "—", str(e).splitlines()[0])

## Step 6 — Stop a loop: a rule, a budget and the recursion limit

[3. Shape the workflow](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-09-fable/03-shape-the-workflow/)

In [ ]:
# A loop needs its own stop condition; the recursion limit is only a last
# resort.
from langgraph.errors import GraphRecursionError
from langgraph._internal._config import DEFAULT_RECURSION_LIMIT

class Draft(TypedDict, total=False):
    question: str
    answer: str
    attempts: int
    spent: float            # dollars spent on model calls in this run
    outcome: str

# illustrative: 1,500 tokens in, 200 out, at gpt-6-luna prices
COST_PER_CALL = 1500 * 0.10 / 1e6 + 200 * 0.50 / 1e6

def build_loop(model, stop_rule):
    def draft(s):
        reply = model.invoke([
            ("human", f"{as_context(retrieve(s['question']))}\n\nQuestion: "
             f"{s['question']}")
        ]).content
        return {
            "answer": reply, "attempts": s.get("attempts", 0) + 1,
            "spent": s.get("spent", 0) + COST_PER_CALL
        }
    # the answer must cite a section like [4.2]; the scripted model never does
    def check(s):
        return {
            "outcome": "ok"
            if re.search(r"\[\d\.\d\]", s["answer"]) else "retry"
        }
    def give_up(s):
        return {"outcome": "handed to a tutor"}
    def after_check(s):
        if s["outcome"] == "ok":
            return END
        return "give_up" if stop_rule(s) else "draft"
    g = StateGraph(Draft)
    g.add_node(draft); g.add_node(check); g.add_node(give_up)
    g.add_edge(START, "draft"); g.add_edge("draft", "check")
    g.add_conditional_edges("check", after_check, ["draft", "give_up", END])
    g.add_edge("give_up", END)
    return g.compile()

q = {
    "question": "Which lecture explains why a cheaper model can cost more "
    "per correct answer?"
}
print("default recursion limit in this version:", int(DEFAULT_RECURSION_LIMIT))

for label, rule, config in [
    ("no stop rule, default limit", lambda s: False, {}),
    ("no stop rule, limit 10", lambda s: False, {"recursion_limit": 10}),
    ("stop after 3 attempts", lambda s: s["attempts"] >= 3, {}),
    ("stop at a $0.0005 budget", lambda s: s["spent"] + COST_PER_CALL > 0.0005,
     {}),
]:
    model, t0 = ScriptedModel(), time.time()
    try:
        out = build_loop(model, rule).invoke(q, config)
        result = out["outcome"]
    except GraphRecursionError:
        result = "GraphRecursionError"
    print(f"{label:<30} {model.calls:>5} model calls  "
          f"${model.calls * COST_PER_CALL:.4f} (illustrative)"
          f"  {time.time() - t0:5.1f} s  → {result}")

## Step 7 — Deterministic checks over the whole assistant

['4. Define what "works" means'](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-09-fable/04-define-what-works-means/)

In [ ]:
# Deterministic checks first: routing, retrieval, tool calls and trajectories,
# all without a judge model.
from pydantic import ValidationError

# Outcome: did the router choose the right lecture? (145 validation questions,
# the scripted router)
router_model = ScriptedModel()
route_graph = build_qa_graph(router_model).compile()
def route_offline(question):
    return route_graph.invoke({"question": question},
                              interrupt_after=["route"])["route"]
t0 = time.time()
routes = [route_offline(q["question"]) for q in val]
route_pass = [r == q["lecture"] for r, q in zip(routes, val)]
print(f"routing: {sum(route_pass)}/{len(val)} correct "
      f"({sum(route_pass) / len(val):.2f}), "
      f"{router_model.calls / len(val):.0f} model call per question, "
      f"{time.time() - t0:.1f} s")

# Step: did retrieval bring the gold section? Recall@5 and MRR on the questions
# that have one.
def rank_of_gold(q, k=5):
    ids = [n["id"] for n in retrieve(q["question"], k)]
    return ids.index(q["section"]) + 1 if q["section"] in ids else None
answerable = [q for q in val if q["lecture"]]
ranks = [rank_of_gold(q) for q in answerable]
print(f"retrieval: recall@5 "
      f"{sum(r is not None for r in ranks) / len(ranks):.2f}, "
      f"MRR {sum(1 / r for r in ranks if r) / len(ranks):.2f} on "
      f"{len(answerable)} questions")

# Step: is each tool call well formed, the right tool, with the right
# arguments?
class DeleteTodo(BaseModel):
    id: int
class SendReminder(BaseModel):
    todo: int
    at: str = Field(pattern=r"^\d{2}:\d{2}$")
ARGS = {"delete_todo": DeleteTodo, "send_reminder": SendReminder}
tool_cases = [  # request, the call expected, the call a model made
    ("Delete the bread task (id 2)", ("delete_todo", {"id": 2}),
     ("delete_todo", {"id": 2})),
    ("Delete the bread task (id 2)", ("delete_todo", {"id": 2}),
     ("delete_todo", {"id": "the bread one"})),
    ("Delete the bread task (id 2)", ("delete_todo", {"id": 2}),
     ("delete_todo", {"id": 3})),
    ("Remind me at 9 about task 3", ("send_reminder",
                                     {"todo": 3, "at": "09:00"}),
     ("send_reminder", {"todo": 3, "at": "9am"})),
    ("Remind me at 9 about task 3",
     ("send_reminder", {"todo": 3,
                        "at": "09:00"}), ("add_todo", {"text": "quiz at 9"})),
]
print(f"{'request':<30}{'schema':<9}{'tool':<7}{'arguments'}")
for request, (want_tool, want_args), (tool, args) in tool_cases:
    try:
        schema_ok = tool in ARGS and bool(ARGS[tool](**args))
    except ValidationError:
        schema_ok = False
    print(
        f"{request:<30}{'ok' if schema_ok else 'FAIL':<9}"
        f"{'ok' if tool == want_tool else 'FAIL':<7}"
        f"{'ok' if (tool, args) == (want_tool, want_args) else 'FAIL'}")

# Trajectory: the steps a run took, compared with the expected ones in four
# modes.
def match(actual, expected, mode):
    a, e = Counter(actual), Counter(expected)
    return {"strict": actual == expected,          # same steps, same order
            "unordered": a == e,                   # same steps, any order
            "subset": not (a - e),  # nothing beyond the expected steps
            # every expected step, extras allowed
            "superset": not (e - a)}[mode]
# our own rule: every approval-level tool comes right after a review
def policy_ok(actual):
    return all(i > 0 and actual[i - 1] == "review"
               for i, s in enumerate(actual)
               if POLICY.get(s) == "approve") \
        and all(n == 1 for s, n in Counter(actual).items()
                if POLICY.get(s) == "approve")
expected = ["search_notes", "check_calendar", "review", "send_reminder"]
trajectories = {
    "as expected": [
        "search_notes", "check_calendar", "review", "send_reminder"
    ],
    "two look-ups swapped": [
        "check_calendar", "search_notes", "review", "send_reminder"
    ],
    "an extra search": [
        "search_notes", "search_notes", "check_calendar", "review",
        "send_reminder"
    ],
    "skipped the calendar": ["search_notes", "review", "send_reminder"],
    "sent without review": ["search_notes", "check_calendar", "send_reminder"],
    "review after sending": [
        "search_notes", "check_calendar", "send_reminder", "review"
    ],
    "sent twice": [
        "search_notes", "check_calendar", "review", "send_reminder",
        "send_reminder"
    ],
}
modes = ["strict", "unordered", "subset", "superset"]
print(f"{'trajectory':<24}" + "".join(f"{m:<10}"
                                      for m in modes) + "policy rule")
for name, actual in trajectories.items():
    print(f"{name:<24}" +
          "".join(f"{'pass' if match(actual, expected, m) else 'FAIL':<10}"
                  for m in modes) + ("pass" if policy_ok(actual) else "FAIL"))

suite_baseline = {"routing": sum(route_pass) / len(val),
                  "recall@5": sum(r is not None for r in ranks) / len(ranks)}
pathlib.Path("suite-baseline.json").write_text(json.dumps(suite_baseline))

## Step 8 — Measure the judge against known labels 💳 API

[5. Judge with a model, and check the judge](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-09-fable/05-judge-with-a-model-and-check-the-judge/)

In [ ]:
# 💳 A model judge for faithfulness, measured on answers whose labels are
# known before it runs.
from langchain_core.callbacks import get_usage_metadata_callback

class Verdict(BaseModel):
    unsupported: list[str] = Field(
        description=
        "every claim in the answer that the notes do not support, copied "
        "word for word")
    verdict: Literal["faithful", "unfaithful"]
RUBRIC = ("You check an answer against course notes. List every claim in the "
          "answer that the notes do not support. "
          "A claim that is true in general but absent from the "
          "notes is unsupported. "
          "The verdict is 'faithful' only if the list is empty.")

random.seed(9)
judge_qs = random.sample([q for q in val if q["lecture"]], 60)
contexts = [as_context(retrieve(q["question"])) for q in judge_qs]

def plain_sentence(note):
    return next(
        (x for x in re.split(r"(?<=[.!?])\s+", note["text"])
         if 8 <= len(x.split()) <= 30 and x[0].isupper() and not set("|·#$`")
         & set(x)), None)
# a true sentence from a lecture none of the retrieved notes belongs to
def foreign_claim(i):
    shown = {n["lecture"] for n in retrieve(judge_qs[i]["question"])}
    others = [
        n for n in notes if n["lecture"] not in shown and plain_sentence(n)
    ]
    return plain_sentence(random.Random(i).choice(others))
# a specific origin story for the topic, in none of the notes
def invented_claim(i):
    topic = re.sub(r"^\d+\.\d+\s*", "",
                   retrieve(judge_qs[i]["question"])[0]["heading"]).lower()
    return (f"The notes add that {topic} was first proposed by a team at "
            f"Google in 2019.")
assert not any("2019" in c and "Google" in c for c in contexts)

STOP = set(
    "that this with from their which these there about into than then they "
    "them have been what when where "
    "will would could should also more most such only each other some".split())
def content_words(text):
    return {w for w in re.findall(r"[a-z]{4,}", text.lower()) if w not in STOP}
# the deterministic baseline: share of the answer's words found in the notes
def word_overlap(answer, context):
    words = content_words(answer)
    return len(words & content_words(context)) / max(1, len(words))

# Cohen's kappa: agreement corrected for the agreement chance alone gives
def kappa(a, b):
    labels = set(a) | set(b)
    seen = sum(x == y for x, y in zip(a, b)) / len(a)
    chance = sum((a.count(l) / len(a)) * (b.count(l) / len(b)) for l in labels)
    return (seen - chance) / (1 - chance)

def in_slices(label, runnable, inputs, size=20):
    out, t0 = [], time.time()
    for i in range(0, len(inputs), size):
        part = runnable.batch(inputs[i:i + size],
                              config={"max_concurrency": 6},
                              return_exceptions=True)
        out += [None if isinstance(p, Exception) else p for p in part]
        print(
            f"    {label}: {len(out)}/{len(inputs)} in "
            f"{time.time() - t0:.0f} s", flush=True)
    return out

if LLM_READY:
    with get_usage_metadata_callback() as usage:
        answers = in_slices("answers", hosted, [[
            ("system",
             "Answer the student's question in at most three sentences, using "
             "only the notes."),
            ("human", f"Notes:\n{c}\n\nQuestion: {q['question']}")
        ] for q, c in zip(judge_qs, contexts)])
        # .text joins the reply's content blocks
        answers = [a.text if a else "" for a in answers]
        cases = []                     # (kind, answer, context, known label)
        for i, (a, c) in enumerate(zip(answers, contexts)):
            cases.append(("original", a, c, "faithful"))
            extra = foreign_claim(i) if i % 2 == 0 else invented_claim(i)
            cases.append(("foreign" if i % 2 == 0 else "invented",
                          f"{a} {extra}", c, "unfaithful"))
        judge = hosted.with_structured_output(Verdict)
        verdicts = in_slices("judge", judge,
                             [[("system", RUBRIC),
                               ("human", f"Notes:\n{c}\n\nAnswer:\n{a}")]
                              for _, a, c, _ in cases])
    judged = [v.verdict if v else "error" for v in verdicts]
    baseline = [
        "faithful" if word_overlap(a, c) >= 0.8 else "unfaithful"
        for _, a, c, _ in cases
    ]
    truth = [label for *_, label in cases]
    print(
        f"{'answers':<12}{'n':>4}{'judge right':>13}"
        f"{'overlap ≥ 0.8 right':>22}"
    )
    for kind in ("original", "foreign", "invented"):
        idx = [i for i, c in enumerate(cases) if c[0] == kind]
        print(
            f"{kind:<12}{len(idx):>4}"
            f"{sum(judged[i] == truth[i] for i in idx):>13}"
            f"{sum(baseline[i] == truth[i] for i in idx):>22}")
    print(f"agreement with the known labels: judge "
          f"{sum(j == t for j, t in zip(judged, truth)) / len(truth):.2f} "
          f"(kappa {kappa(judged, truth):.2f}), word overlap "
          f"{sum(b == t for b, t in zip(baseline, truth)) / len(truth):.2f} "
          f"(kappa {kappa(baseline, truth):.2f})")
    flagged = [(cases[i][1], verdicts[i].unsupported)
               for i in range(len(cases))
               if cases[i][0] == "original" and judged[i] == "unfaithful"]
    print(f"originals the judge called unfaithful: {len(flagged)}; the first "
          f"two, to read by hand:")
    for answer, claims in flagged[:2]:
        print("   answer:", answer[:160], "\n   judge's unsupported claims:",
              claims)
    u = usage.usage_metadata.get(MODEL, {}) or next(
        iter(usage.usage_metadata.values()), {})
    print(
        f"{len(answers) + len(cases)} calls, {u.get('input_tokens', 0):,} "
        f"tokens in, {u.get('output_tokens', 0):,} out, "
        + "$" + format((u.get('input_tokens', 0) * PRICE_IN
                        + u.get('output_tokens', 0) * PRICE_OUT) / 1e6, ".3f")
    )

## Step 9 — Compare two answers in both orders 💳 API

[5. Judge with a model, and check the judge](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-09-fable/05-judge-with-a-model-and-check-the-judge/)

In [ ]:
# 💳 Pairwise: which of two answers is better? Each pair is judged in both
# orders.
class Preference(BaseModel):
    better: Literal["A", "B"]
PAIR_RUBRIC = (
    "Two answers to a student's question, and the course notes. Which answer "
    "is better: correct "
    "according to the notes, complete and clear? Reply A or B.")
PADDING = (
    " This is an important idea in the course, and understanding it well "
    "will make the later lectures "
    "much easier to follow.")

# first: the verdict with the original as A; second: with it as B
def outcome(first, second):
    if first == "A" and second == "B":
        return "original won both orders"
    if first == "B" and second == "A":
        return "copy won both orders"
    return ("slot A won both orders" if first == "A"
            else "slot B won both orders")

if LLM_READY:
    # x = the original answer, y = a copy with one unsupported claim, or with
    # an empty sentence added
    pair_sets = {
        "original vs invented claim": [(cases[2 * i][1], cases[2 * i + 1][1],
                                        contexts[i], judge_qs[i])
                                       for i in range(1, 60, 2)],
        "original vs padded copy": [(answers[i], answers[i] + PADDING,
                                     contexts[i], judge_qs[i])
                                    for i in range(30)],
    }
    pick = hosted.with_structured_output(Preference)
    def ask(a, b, c, q):
        return [
            ("system", PAIR_RUBRIC),
            ("human",
             f"Notes:\n{c}\n\nQuestion: {q['question']}\n\nA: {a}\n\nB: {b}")
        ]
    with get_usage_metadata_callback() as usage:
        for name, pairs in pair_sets.items():
            ab = in_slices(f"{name}, x first", pick,
                           [ask(x, y, c, q) for x, y, c, q in pairs])
            ba = in_slices(f"{name}, y first", pick,
                           [ask(y, x, c, q) for x, y, c, q in pairs])
            results = Counter(
                outcome(p.better, r.better) for p, r in zip(ab, ba) if p and r)
            print(f"{name} ({len(pairs)} pairs):",
                  ", ".join(f"{k} {v}" for k, v in results.most_common()))
    u = next(iter(usage.usage_metadata.values()), {})
    print(
        f"{4 * 30} calls, "
        + "$" + format((u.get('input_tokens', 0) * PRICE_IN
                        + u.get('output_tokens', 0) * PRICE_OUT) / 1e6, ".3f")
    )

## Step 10 — Run the unchanged router three times 💳 API

[6. Keep it working](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-09-fable/06-keep-it-working/)

In [ ]:
# 💳 The noise baseline: the same router, same prompt, same settings, run
# three times on the 145 questions.
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

# Lecture 5's router, unchanged: all section headings, three retrieved notes, a
# strict schema.
NAMES = {
    1: "Generative and agentic AI", 2: "Training language models",
    3: "Choosing and configuring models", 4: "Context engineering",
    5: "LangChain", 6: "Agents and tools"
}
headings = defaultdict(list)
for n in notes:
    headings[n["lecture"]].append(n["heading"])
HEADINGS = "\n".join(f"{k}: {v}\n  " + "\n  ".join(headings[k])
                     for k, v in NAMES.items())
TASK = (
    "You route a student's question to the lecture of the course notes "
    "that answers it. "
    "Return lecture 0 if no lecture covers it.\n\nLectures:\n")
def with_notes(question, k=3):
    return "\n\n".join(f"[Lecture {n['lecture']}] "
                       f"{n['heading']}\n{' '.join(n['text'].split()[:250])}"
                       for n in retrieve(question, k))
route_prompt = ChatPromptTemplate.from_messages([
    ("system", TASK + "{headings}"),
    ("human", "Notes that may help:\n{notes}\n\nQuestion: {question}")])
def make_router(model):
    return (RunnablePassthrough.assign(
        headings=lambda q: HEADINGS, notes=lambda q: with_notes(q["question"]))
            | route_prompt | model.with_structured_output(Route)
            | RunnableLambda(lambda r: r.lecture))

if LLM_READY:
    router = make_router(hosted)
    with get_usage_metadata_callback() as usage:
        runs_of = [
            in_slices(f"run {r + 1}", router, val, size=50) for r in range(3)
        ]
    acc = [
        sum(p == q["lecture"] for p, q in zip(run, val)) / len(val)
        for run in runs_of
    ]
    changed = [
        i for i in range(len(val)) if len({run[i]
                                           for run in runs_of}) > 1
    ]
    print("accuracy per run:", ", ".join(f"{a:.3f}" for a in acc),
          f"| spread {max(acc) - min(acc):.3f}")
    print(f"questions whose route changed between runs: "
          f"{len(changed)} of {len(val)}")
    for i in changed[:3]:
        print(f"   {val[i]['question'][:70]:<72} correct {val[i]['lecture']}, "
              f"runs {[run[i] for run in runs_of]}")
    noise = {"runs": acc, "changed": len(changed), "n": len(val)}
    pathlib.Path("noise-baseline.json").write_text(json.dumps(noise))
    u = next(iter(usage.usage_metadata.values()), {})
    print(
        f"{3 * len(val)} calls, "
        + "$" + format((u.get('input_tokens', 0) * PRICE_IN
                        + u.get('output_tokens', 0) * PRICE_OUT) / 1e6, ".3f")
    )

## Step 11 — Run the suite as a regression test, then sample production

[6. Keep it working](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-09-fable/06-keep-it-working/)

In [ ]:
# The suite as a regression test: pytest runs it on each candidate change;
# production failures become new cases.
import pytest

# fixed by the action's nature, not by a setting
IRREVERSIBLE = {name for name, t in TOOLS.items() if not t["undo"]}
def trajectory_under(policy, calls):
    steps = []
    for name, _ in calls:
        rule = policy.get(name, "approve")
        steps += [
            "review", name
        ] if rule == "approve" else ["draft"] if rule == "suggest" else [name]
    return steps
def recall_at_5(retrieve_fn):
    return sum(
        any(n["id"] == q["section"] for n in retrieve_fn(q["question"], 5))
        for q in answerable) / len(answerable)
# the scripted router follows the first retrieved note
def routing_accuracy(retrieve_fn, cases):
    return sum(
        retrieve_fn(q["question"], 1)[0]["lecture"] == q["lecture"]
        for q in cases) / len(cases)

noise = json.loads(pathlib.Path("noise-baseline.json").read_text()
                   ) if pathlib.Path("noise-baseline.json").exists() else None
if noise:
    print(f"hosted router: {noise['changed']} of {noise['n']} routes changed "
          f"between identical runs, so a change "
          f"must move accuracy by more than "
          f"{noise['changed'] / noise['n']:.3f} to count")

TESTS = '''
"""Regression suite of the study assistant. In a project, import from its
package instead of __main__."""
import __main__ as app

def test_routing_not_worse():
    assert (app.routing_accuracy(app.candidate["retrieve"], app.val)
            >= app.BASELINE["routing"] - app.MARGIN)

def test_retrieval_not_worse():
    assert (app.recall_at_5(app.candidate["retrieve"])
            >= app.BASELINE["recall@5"] - app.MARGIN)

def test_irreversible_actions_are_reviewed():
    for request, calls in app.asked:
        steps = app.trajectory_under(app.candidate["policy"], calls)
        for i, step in enumerate(steps):
            if step in app.IRREVERSIBLE:
                assert i > 0 and steps[i - 1] == "review", \
                    f"{request}: {steps}"

def test_reminder_sent_once_after_a_retry():
    assert app.run_retry(with_key=True)[0] == 1
'''
pathlib.Path("test_assistant.py").write_text(TESTS)

# the scripted model gives the same answer every run, so no drop is noise
MARGIN = 0.0
BASELINE = {
    "routing": routing_accuracy(retrieve, val),
    "recall@5": recall_at_5(retrieve)
}
print(f"baseline: routing {BASELINE['routing']:.3f} (Step 7: "
      f"{suite_baseline['routing']:.3f}), "
      f"recall@5 {BASELINE['recall@5']:.3f}")

bigram = TfidfVectorizer(stop_words="english",
                         ngram_range=(1, 2)).fit([n["text"] for n in notes])
bigram_m = bigram.transform([n["text"] for n in notes])
def retrieve_bigrams(question, k=3):
    scores = (bigram_m @ bigram.transform([question]).T).toarray().ravel()
    return [notes[i] for i in scores.argsort()[::-1][:k]]

candidates = {
    "no change": {"retrieve": retrieve, "policy": POLICY},
    "retrieval with word pairs": {
        "retrieve": retrieve_bigrams, "policy": POLICY
    },
    "deletions without approval": {
        "retrieve": retrieve, "policy": {**POLICY, "delete_todo": "report"}
    },
}
class Report:
    def __init__(self):
        self.failed = []
    def pytest_runtest_logreport(self, report):
        if report.when == "call" and report.failed:
            self.failed.append(report.nodeid.split("::")[1])
# the tests read `candidate` from this session
for name, candidate in candidates.items():
    report = Report()
    code = pytest.main([
        "-qq", "--tb=no", "-rN", "--color=no", "-p", "no:warnings", "-p",
        "no:cacheprovider", "test_assistant.py"
    ], plugins=[report])
    print(f"=> {name}: "
          f"{'PASS' if code == 0 else 'FAIL ' + ', '.join(report.failed)}"
          f" | routing {routing_accuracy(candidate['retrieve'], val):.3f}, "
          f"recall@5 {recall_at_5(candidate['retrieve']):.3f}")

# Online: a sample of "production" questions, with no labels. Two signals flag
# a request for review.
def top_score(question):
    return (doc_m @ vec.transform([question]).T).toarray().ravel().max()
random.seed(11)
traffic = random.sample(train, 300)
flagged, missed = [], 0
for q in traffic:
    top3 = retrieve(q["question"], 3)
    disagree = Counter(
        n["lecture"] for n in top3).most_common(1)[0][0] != top3[0]["lecture"]
    weak = top_score(q["question"]) < 0.15
    # known here; in production a reviewer finds out
    wrong = top3[0]["lecture"] != q["lecture"]
    if disagree or weak:
        flagged.append((q, wrong))
    elif wrong:
        missed += 1
confirmed = [q for q, wrong in flagged if wrong]
print(f"production sample: {len(traffic)} questions; {len(flagged)} flagged "
      f"for review, of which {len(confirmed)} were "
      f"wrong ({len(confirmed) / len(flagged):.0%}); {missed} wrong "
      f"answers went unflagged "
      f"(the signals caught "
      f"{len(confirmed) / (len(confirmed) + missed):.0%} of them)")
# new suite cases; the current router gets all of them wrong
regression_cases = confirmed
print(f"new regression cases: {len(regression_cases)}; a change that fixes "
      f"some of them shows up as a gain on this set")

## Step 12 — The evaluation plan in one table

[6. Keep it working](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-09-fable/06-keep-it-working/)

In [ ]:
# The evaluation suite in one table: what each check catches, what it costs,
# and when it runs.
judge_note = (
    f"agreement {sum(j == t for j, t in zip(judged, truth)) / len(truth):.2f}"
    if LLM_READY else "needs Step 8")
noise_note = (f"{noise['changed']}/{noise['n']} routes changed"
              if noise else "needs Step 10")
rows_ = [
    ("action policy guard", "trajectory", "irreversible action without review",
     "free, < 1 s", "every commit", "Steps 1, 11"),
    ("idempotency key", "outcome", "a reminder sent twice", "free, < 1 s",
     "every commit", "Steps 3, 11"),
    ("routing label match", "outcome",
     f"wrong lecture ({suite_baseline['routing']:.2f} correct)",
     "1 call per question", "every commit / before release", "Step 7"),
    ("recall@5, MRR", "step",
     f"right note not retrieved (recall {suite_baseline['recall@5']:.2f})",
     "free, < 1 s", "every commit", "Step 7"),
    ("tool schema and arguments", "step", "malformed or wrong arguments",
     "free", "every commit", "Step 7"),
    ("trajectory match + policy rule", "trajectory",
     "missing, extra, reordered or repeated steps", "free", "every commit",
     "Step 7"),
    ("model judge (faithfulness)", "outcome",
     f"unsupported claims ({judge_note})", "1 call per answer",
     "before release, on a sample online", "Step 8"),
    ("pairwise comparison", "outcome",
     "which of two versions is better (both orders)", "2 calls per pair",
     "when choosing between versions", "Step 9"),
    ("noise baseline", "—", f"size of a real change ({noise_note})",
     "3 runs of the suite", "when the model or settings change", "Step 10"),
    ("production signals", "outcome", "new failures the suite lacks",
     "review time", "continuously", "Step 11"),
]
print(f"{'check':<31}{'level':<11}{'catches':<50}{'cost':<20}{'runs'}")
for check, level, catches, cost, when, step in rows_:
    print(f"{check:<31}{level:<11}{catches:<50}{cost:<20}{when} ({step})")